# AILabTools 비교용 test 세트 만들기

한국인 피부상태 측정 데이터(스마트폰 사진)에서 test 세트를 만든다.

1. 사진이 있는 피험자(Training + Validation) 중 400명을 무작위로 뽑는다. (seed 고정, 한 번 뽑으면 파일로 고정)
2. 뽑힌 순서대로 정면·좌·우 3장을 AILabTools Skin Analyze Pro 에 한 요청으로 보낸다.
3. 실패한 사람은 기록만 남기고 건너뛴다. 네트워크 오류만 한 번 재시도한다.
4. **성공이 165명(`TARGET`)이 되면 멈춘다.** 이 165명이 test 세트이고, 내 모델 학습에서는 반드시 뺀다.

결과는 `ailabtools/test_set/` 에 남는다.

| 파일 | 내용 |
|---|---|
| `sample_400.json` | 뽑힌 400명과 호출 순서, seed |
| `status.json` | 호출한 사람별 성공/실패, 에러, 응답 시간 |
| `raw/<subject_id>.json` | 성공한 사람의 API 응답 원본 |
| `raw_failed/<subject_id>.json` | 실패한 사람의 API 응답 원본 (응답이 있었던 경우만) |
| `test_subjects_165.json` | 최종 test 세트 165명 |

호출당 과금되므로 `DRY_RUN = True` 가 기본이다. 중간에 멈추거나 커널이 죽어도 다시 실행하면 호출하지 않은 사람부터 이어간다.

In [2]:
import json
import os
import random
import time
from datetime import datetime, timezone
from pathlib import Path

import requests
from dotenv import load_dotenv

# 노트북은 ailabtools/ 에서 실행되고, .env 와 data/ 는 그 상위(프로젝트 루트)에 있다.
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "ailabtools" else Path.cwd()
DATA_DIR = PROJECT_DIR / "data" / "028.한국인_피부상태_측정_데이터" / "3.개방데이터" / "1.데이터"

OUT_DIR = PROJECT_DIR / "ailabtools" / "test_set"
RAW_DIR = OUT_DIR / "raw"
FAILED_DIR = OUT_DIR / "raw_failed"
SAMPLE_PATH = OUT_DIR / "sample_400.json"
STATUS_PATH = OUT_DIR / "status.json"

SEED = 42
SAMPLE_SIZE = 400
TARGET = 165
TEST_SET_PATH = OUT_DIR / f"test_subjects_{TARGET}.json"

SPLITS = ("Training", "Validation")
DEVICE_DIR = "3. 스마트폰"
DEVICE_CODE = "03"
VIEWS = {"image": "F", "left_side_image": "L", "right_side_image": "R"}  # API 필드 -> 사진 각도

load_dotenv(PROJECT_DIR / ".env")
API_KEY = os.getenv("AILABTOOLS_API_KEY")
ENDPOINT = os.getenv("AILABTOOLS_BASE_URL", "https://www.ailabapi.com/api/portrait/analysis/skin-analysis-pro")

MAX_IMAGE_BYTES = 8 * 1024 * 1024  # AILabTools 제한
CONNECT_TIMEOUT = 15
READ_TIMEOUT = 120
SLEEP_BETWEEN_CALLS = 0.5
RETRY_WAIT = 5

STATUS_SUCCESS = "success"
STATUS_FAILED = "failed"

assert DATA_DIR.is_dir(), f"데이터 폴더가 없습니다: {DATA_DIR}"
assert API_KEY, "AILABTOOLS_API_KEY 가 .env 에 없습니다."
for d in (OUT_DIR, RAW_DIR, FAILED_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("프로젝트:", PROJECT_DIR)
print("결과 폴더:", OUT_DIR)

프로젝트: /Users/USER/Desktop/소마/face_analysis_AI
결과 폴더: /Users/USER/Desktop/소마/face_analysis_AI/ailabtools/test_set


## 1. 400명 뽑기
`sample_400.json` 이 이미 있으면 다시 뽑지 않고 그대로 불러온다. 다시 뽑으려면 파일을 직접 지워야 한다.

In [3]:
def now():
    return datetime.now(timezone.utc).isoformat()


def rel(path):
    return str(Path(path).relative_to(PROJECT_DIR))


def subject_photos(split, subject_id):
    folder = DATA_DIR / split / "01.원천데이터" / DEVICE_DIR / subject_id
    return {field: folder / f"{subject_id}_{DEVICE_CODE}_{view}.jpg" for field, view in VIEWS.items()}


def write_json(path, obj):
    # 쓰는 도중 멈춰도 기존 파일이 깨지지 않게 임시 파일에 쓰고 바꿔치기한다.
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    tmp.replace(path)


if SAMPLE_PATH.exists():
    sample = json.loads(SAMPLE_PATH.read_text(encoding="utf-8"))
    print(f"저장된 목록을 불러왔습니다. ({sample['created_at']}, seed={sample['seed']})")
else:
    candidates = []
    for split in SPLITS:
        for subject_id in sorted(os.listdir(DATA_DIR / split / "01.원천데이터" / DEVICE_DIR)):
            if subject_id.isdigit() and all(p.is_file() for p in subject_photos(split, subject_id).values()):
                candidates.append({"subject_id": subject_id, "split": split})

    picked = random.Random(SEED).sample(candidates, SAMPLE_SIZE)
    sample = {
        "seed": SEED,
        "device": DEVICE_DIR,
        "n_candidates": len(candidates),
        "created_at": now(),
        "subjects": [{"order": i, **c} for i, c in enumerate(picked, start=1)],
    }
    write_json(SAMPLE_PATH, sample)
    print(f"후보 {len(candidates)}명 중 {SAMPLE_SIZE}명을 뽑아 저장했습니다.")

split_counts = {s: sum(1 for x in sample["subjects"] if x["split"] == s) for s in SPLITS}
print("split 별 인원:", split_counts)
print("앞 5명:", [x["subject_id"] for x in sample["subjects"][:5]])

후보 965명 중 400명을 뽑아 저장했습니다.
split 별 인원: {'Training': 354, 'Validation': 46}
앞 5명: ['0834', '0142', '0032', '0970', '0361']


## 2. API 호출 함수

In [4]:
def call_api(photos):
    """정면·좌·우 3장을 한 요청에 보낸다. (raw, http_status, status, error, latency_ms) 를 돌려준다."""
    files = {}
    for field, path in photos.items():
        data = path.read_bytes()
        if len(data) > MAX_IMAGE_BYTES:
            # 호출하지 않으므로 과금되지 않는다.
            return None, None, STATUS_FAILED, f"이미지 8MB 초과: {path.name} ({len(data):,} bytes)", 0
        files[field] = (path.name, data, "image/jpeg")

    started = time.monotonic()
    try:
        response = requests.post(
            ENDPOINT,
            headers={"ailabapi-api-key": API_KEY},
            files=files,
            timeout=(CONNECT_TIMEOUT, READ_TIMEOUT),
        )
    except requests.RequestException as exc:
        return None, None, STATUS_FAILED, f"네트워크 오류: {exc}", int((time.monotonic() - started) * 1000)

    latency_ms = int((time.monotonic() - started) * 1000)
    try:
        raw = response.json()
    except ValueError:
        return None, response.status_code, STATUS_FAILED, f"JSON 이 아닌 응답: {response.text[:500]}", latency_ms

    # 얼굴 미검출은 HTTP 422 로 온다. 이때도 본문은 JSON 이라 raw 로 남긴다.
    if response.status_code != 200:
        detail = raw.get("error_detail") or raw.get("error_msg") or ""
        if isinstance(detail, dict):
            detail = detail.get("code_message") or detail.get("code") or ""
        return raw, response.status_code, STATUS_FAILED, f"HTTP {response.status_code} {detail}".strip(), latency_ms

    error_code = raw.get("error_code")
    if error_code != 0 or not raw.get("result"):
        detail = raw.get("error_detail") or raw.get("error_msg") or ""
        return raw, response.status_code, STATUS_FAILED, f"error_code={error_code} {detail}".strip(), latency_ms

    return raw, response.status_code, STATUS_SUCCESS, None, latency_ms

## 3. 순서대로 호출 — 성공 165명에서 멈춤
처음에는 `DRY_RUN = True` 로 실행해 대상을 확인하고, 실제로 호출할 때 `False` 로 바꾼다.

- 이미 기록된 사람(성공·실패 모두)은 다시 호출하지 않는다.
- 결과는 한 명 끝날 때마다 바로 저장한다. 멈추려면 셀 실행을 중단(■)하면 된다.

In [6]:
DRY_RUN = False

status = json.loads(STATUS_PATH.read_text(encoding="utf-8")) if STATUS_PATH.exists() else {}


def n_success():
    return sum(1 for r in status.values() if r["status"] == STATUS_SUCCESS)


pending = [s for s in sample["subjects"] if s["subject_id"] not in status]
print(f"성공 {n_success()}/{TARGET}명 · 실패 {len(status) - n_success()}명 · 아직 호출 안 함 {len(pending)}명")

if n_success() >= TARGET:
    print(f"이미 {TARGET}명을 모두 확보했습니다.")
elif DRY_RUN:
    need = TARGET - n_success()
    print(f"\n성공 {need}명이 더 필요합니다. 다음 호출 순서:")
    for s in pending[:10]:
        print(f"  #{s['order']:>3}  {s['subject_id']}  ({s['split']})")
    print("\nDRY_RUN = True 라서 호출하지 않았습니다. 호출당 과금됩니다.")
else:
    try:
        for s in pending:
            if n_success() >= TARGET:
                break
            subject_id = s["subject_id"]
            photos = subject_photos(s["split"], subject_id)

            raw, http_status, result, error, latency_ms = call_api(photos)
            retried = False
            if error and error.startswith("네트워크 오류"):
                time.sleep(RETRY_WAIT)
                raw, http_status, result, error, latency_ms = call_api(photos)
                retried = True

            if raw is not None:
                write_json((RAW_DIR if result == STATUS_SUCCESS else FAILED_DIR) / f"{subject_id}.json", raw)

            status[subject_id] = {
                "order": s["order"],
                "subject_id": subject_id,
                "split": s["split"],
                "status": result,
                "http_status": http_status,
                "error": error,
                "retried": retried,
                "latency_ms": latency_ms,
                "photos": {field: rel(p) for field, p in photos.items()},
                "called_at": now(),
            }
            write_json(STATUS_PATH, status)

            if result == STATUS_SUCCESS:
                print(f"#{s['order']:>3} {subject_id}  성공 ({latency_ms}ms)  → {n_success()}/{TARGET}")
            else:
                print(f"#{s['order']:>3} {subject_id}  실패 — {error}")

            if n_success() < TARGET:
                time.sleep(SLEEP_BETWEEN_CALLS)
    except KeyboardInterrupt:
        print("\n중단했습니다. 여기까지의 결과는 저장돼 있고, 다시 실행하면 이어서 호출합니다.")

    print(f"\n성공 {n_success()}/{TARGET}명 · 실패 {len(status) - n_success()}명")
    if n_success() < TARGET and all(s["subject_id"] in status for s in sample["subjects"]):
        print(f"{SAMPLE_SIZE}명을 모두 호출했는데 {TARGET}명을 채우지 못했습니다.")

성공 0/165명 · 실패 0명 · 아직 호출 안 함 400명
#  1 0834  성공 (5522ms)  → 1/165
#  2 0142  성공 (5967ms)  → 2/165
#  3 0032  성공 (5082ms)  → 3/165
#  4 0970  성공 (5235ms)  → 4/165
#  5 0361  성공 (6624ms)  → 5/165
#  6 0322  성공 (6064ms)  → 6/165
#  7 0296  성공 (6008ms)  → 7/165
#  8 0180  성공 (5527ms)  → 8/165
#  9 0962  성공 (4778ms)  → 9/165
# 10 0132  성공 (5258ms)  → 10/165
# 11 0883  성공 (6135ms)  → 11/165
# 12 0968  성공 (6228ms)  → 12/165
# 13 0614  성공 (7978ms)  → 13/165
# 14 0714  성공 (5967ms)  → 14/165
# 15 0114  성공 (5731ms)  → 15/165
# 16 0772  성공 (5256ms)  → 16/165
# 17 0557  성공 (4774ms)  → 17/165
# 18 0040  성공 (5408ms)  → 18/165
# 19 0038  성공 (6210ms)  → 19/165
# 20 0123  성공 (6948ms)  → 20/165
# 21 0289  성공 (6096ms)  → 21/165
# 22 0308  성공 (7196ms)  → 22/165
# 23 0663  성공 (7026ms)  → 23/165
# 24 0787  성공 (4369ms)  → 24/165
# 25 0035  성공 (5032ms)  → 25/165
# 26 0734  성공 (8034ms)  → 26/165
# 27 0263  성공 (6140ms)  → 27/165
# 28 0933  성공 (4316ms)  → 28/165
# 29 0847  성공 (5691ms)  → 29/165
# 30 0916  실패 — H

## 4. test 세트 165명 저장
성공한 사람을 호출 순서대로 `test_subjects_165.json` 에 저장한다. 165명이 안 되면 저장하지 않는다.

In [7]:
status = json.loads(STATUS_PATH.read_text(encoding="utf-8")) if STATUS_PATH.exists() else {}
succeeded = sorted((r for r in status.values() if r["status"] == STATUS_SUCCESS), key=lambda r: r["order"])
failed = [r for r in status.values() if r["status"] == STATUS_FAILED]

missing_raw = [r["subject_id"] for r in succeeded if not (RAW_DIR / f"{r['subject_id']}.json").is_file()]
assert not missing_raw, f"응답 원본 파일이 없는 성공 기록: {missing_raw}"

print(f"성공 {len(succeeded)}명 · 실패 {len(failed)}명")
for r in failed:
    print(f"  실패 #{r['order']:>3} {r['subject_id']}: {r['error']}")

if len(succeeded) < TARGET:
    print(f"\n아직 {TARGET - len(succeeded)}명이 부족해서 test 세트를 저장하지 않았습니다.")
else:
    test_set = {
        "seed": sample["seed"],
        "device": sample["device"],
        "model": "ailabtools-skin-analysis-pro",
        "endpoint": ENDPOINT,
        "n_subjects": len(succeeded),
        "split_counts": {s: sum(1 for r in succeeded if r["split"] == s) for s in SPLITS},
        "created_at": now(),
        "subjects": [
            {
                "subject_id": r["subject_id"],
                "split": r["split"],
                "order": r["order"],
                "photos": r["photos"],
                "raw": rel(RAW_DIR / f"{r['subject_id']}.json"),
            }
            for r in succeeded
        ],
    }
    write_json(TEST_SET_PATH, test_set)
    print(f"\n저장했습니다: {rel(TEST_SET_PATH)}")
    print("split 별 인원:", test_set["split_counts"])

성공 165명 · 실패 2명
  실패 # 30 0916: HTTP 422 Unable to recognize face.
  실패 # 75 0754: HTTP 422 File resolution exceeds limits.

저장했습니다: ailabtools/test_set/test_subjects_165.json
split 별 인원: {'Training': 144, 'Validation': 21}
